In [1]:
"""
Deep Learning Project 1 — Modified ResNet on CIFAR-10 (< 5M parameters)

Architecture: 3-stage ResNet, BasicBlocks, channels (64, 128, 256), blocks (3, 4, 3)
              -> ~4.7M trainable parameters (asserted < 5M below)
Training    : SGD + Nesterov, OneCycle LR, label smoothing, AutoAugment + Cutout,
              mixed precision, horizontal-flip TTA at inference.

Usage:
    python train.py --epochs 200 --kaggle-dir ./kaggle_data
Outputs (in --out):
    best_model.pt, history.json, curves.png, submission.csv

Reference: ResNet code structure adapted from kuangliu/pytorch-cifar
(https://github.com/kuangliu/pytorch-cifar), He et al., "Deep Residual Learning
for Image Recognition", CVPR 2016.
"""
import argparse, json, os, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


# --------------------------------------------------------------------------- #
# Model
# --------------------------------------------------------------------------- #
class BasicBlock(nn.Module):
    """ReLU(S(x) + F(x)),  F = conv-BN-ReLU-conv-BN,  S = identity or 1x1 conv-BN."""

    def __init__(self, in_c, out_c, stride=1, kernel=3, skip_kernel=1):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv2d(in_c, out_c, kernel, stride, pad, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel, 1, pad, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, skip_kernel, stride, skip_kernel // 2, bias=False),
                nn.BatchNorm2d(out_c),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNetSmall(nn.Module):
    """
    N  = number of residual layers           = len(channels)
    Bi = residual blocks in layer i          = blocks[i]
    Ci = channels in layer i                 = channels[i]
    Fi = conv kernel size                    = 3
    Ki = skip-connection kernel size         = 1
    P  = average pool                        = global (8x8 -> 1x1)
    """

    def __init__(self, channels=(64, 128, 256), blocks=(3, 3, 3), num_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(inplace=True),
        )
        layers, in_c = [], channels[0]
        for i, (c, b) in enumerate(zip(channels, blocks)):
            stride = 1 if i == 0 else 2
            for j in range(b):
                layers.append(BasicBlock(in_c, c, stride if j == 0 else 1))
                in_c = c
        self.layers = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(in_c, num_classes)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = torch.flatten(self.pool(x), 1)
        return self.fc(self.drop(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# --------------------------------------------------------------------------- #
# Data
# --------------------------------------------------------------------------- #
class Cutout:
    def __init__(self, size=16):
        self.size = size

    def __call__(self, img):  # img: tensor C,H,W
        h, w = img.shape[1:]
        y, x = np.random.randint(h), np.random.randint(w)
        y1, y2 = max(0, y - self.size // 2), min(h, y + self.size // 2)
        x1, x2 = max(0, x - self.size // 2), min(w, x + self.size // 2)
        img[:, y1:y2, x1:x2] = 0
        return img


class ArrayDataset(torch.utils.data.Dataset):
    """Images held in memory as uint8 HWC numpy arrays."""

    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        return self.transform(Image.fromarray(self.images[i])), int(self.labels[i])


def load_pngs(folder, filenames):
    return np.stack([np.array(Image.open(os.path.join(folder, f)).convert("RGB")) for f in filenames])


def load_kaggle(data_dir):
    """Kaggle layout: train_labels.csv (id,filename,label), train_images/, test.csv (id,filename), test_images/.
    The decoded images are cached to data_cache.npz on first load, so later runs start in seconds."""
    cache = os.path.join("/kaggle/working", "cifar_data_cache.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        print(f"Loaded cached arrays from {cache}", flush=True)
        return d["x_tr"], d["y_tr"], d["x_te"], d["ids"]
    tr = pd.read_csv(os.path.join(data_dir, "train_labels.csv"))
    te = pd.read_csv(os.path.join(data_dir, "test.csv"))
    print("Reading PNGs (first run only, can take a few minutes)...", flush=True)
    x_tr = load_pngs(os.path.join(data_dir, "train_images"), tr["filename"])
    x_te = load_pngs(os.path.join(data_dir, "test_images"), te["filename"])
    y_tr, ids = tr["label"].values.astype(np.int64), te["id"].values
    try:
        np.savez(cache, x_tr=x_tr, y_tr=y_tr, x_te=x_te, ids=ids)
    except OSError as e:
        print("(could not write cache:", e, ")")
    print(f"Loaded train {x_tr.shape}, test {x_te.shape}", flush=True)
    return x_tr, y_tr, x_te, ids


TRAIN_TF = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
        T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
TEST_TF = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])


# --------------------------------------------------------------------------- #
# Train / eval
# --------------------------------------------------------------------------- #
def _shift(x, dx, dy):
    """Translate a batch by (dx, dy) pixels with reflect padding."""
    if dx == 0 and dy == 0:
        return x
    p = max(abs(dx), abs(dy))
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    h, w = x.shape[2:]
    return xp[:, :, p + dy:p + dy + h, p + dx:p + dx + w]


TTA_VIEWS = [(0, 0), (2, 0), (-2, 0), (0, 2), (0, -2)]   # x each with/without horizontal flip


@torch.no_grad()
def predict_logits(model, x, device, tta="full"):
    """tta: 'none' | 'flip' (2 views) | 'full' (10 views: 5 shifts x flip). Returns mean softmax probs."""
    model.eval()
    views = [(0, 0)] if tta in ("none", "flip") else TTA_VIEWS
    flips = [False] if tta == "none" else [False, True]
    probs = 0
    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
        for dx, dy in views:
            xs = _shift(x, dx, dy)
            for fl in flips:
                probs = probs + F.softmax(model(torch.flip(xs, dims=[3]) if fl else xs).float(), 1)
    return probs / (len(views) * len(flips))


@torch.no_grad()
def evaluate(model, loader, device, tta="flip"):
    correct = total = 0
    loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        p = predict_logits(model, x, device, tta)
        loss_sum += F.nll_loss(torch.log(p + 1e-8), y, reduction="sum").item()
        correct += (p.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total


def cutmix(x, y, alpha=1.0):
    """CutMix (Yun et al., 2019): paste a random box from a shuffled batch; returns mixed x, (y_a, y_b, lam)."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    h, w = x.shape[2:]
    rh, rw = int(h * np.sqrt(1 - lam)), int(w * np.sqrt(1 - lam))
    cy, cx = np.random.randint(h), np.random.randint(w)
    y1, y2 = np.clip(cy - rh // 2, 0, h), np.clip(cy + rh // 2, 0, h)
    x1, x2 = np.clip(cx - rw // 2, 0, w), np.clip(cx + rw // 2, 0, w)
    x = x.clone()
    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)
    return x, y, y[idx], lam


def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=240)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--lr", type=float, default=0.1)
    ap.add_argument("--wd", type=float, default=5e-4)
    ap.add_argument("--label-smoothing", type=float, default=0.1)
    ap.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 2))
    ap.add_argument("--data-dir", default="./data", help="folder with train_labels.csv, test.csv, *_images/")
    ap.add_argument("--val-size", type=int, default=5000,
                    help="held-out validation images; use 0 to train on all 50k (final run)")
    ap.add_argument("--out", default="./outputs")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--resume", action="store_true", help="continue from <out>/last.pt (e.g. after a Colab disconnect)")
    ap.add_argument("--mixup-prob", type=float, default=0.5)
    ap.add_argument("--mixup-off-epochs", type=int, default=20)
    ap.add_argument("--ema", type=float, default=0.999, help="EMA decay for weight averaging (0 = off)")
    args = ap.parse_args()

    torch.manual_seed(args.seed); np.random.seed(args.seed)
    os.makedirs(args.out, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True

    model = ResNetSmall().to(device).to(memory_format=torch.channels_last)
    n_params = count_params(model)
    print(f"Trainable parameters: {n_params:,}")
    assert n_params < 5_000_000, "Model exceeds 5M parameter budget!"
    try:
        from torchsummary import summary
        summary(model, (3, 32, 32), device=device.type)
    except Exception as e:
        print("(torchsummary skipped:", e, ")")

    x_all, y_all, x_kaggle, kaggle_ids = load_kaggle(args.data_dir)
    perm = np.random.RandomState(args.seed).permutation(len(x_all))
    val_idx, tr_idx = perm[:args.val_size], perm[args.val_size:]
    kw = dict(num_workers=args.workers, pin_memory=True, persistent_workers=args.workers > 0)
    train_loader = torch.utils.data.DataLoader(
        ArrayDataset(x_all[tr_idx], y_all[tr_idx], TRAIN_TF),
        args.batch_size, shuffle=True, drop_last=True, **kw)
    has_val = args.val_size > 0
    test_loader = torch.utils.data.DataLoader(
        ArrayDataset(x_all[val_idx], y_all[val_idx], TEST_TF), 512, shuffle=False, **kw) if has_val else None
    print(f"Train {len(tr_idx)} | Val {len(val_idx)}")

    # no weight decay on BN / bias
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    opt = torch.optim.SGD([{"params": decay, "weight_decay": args.wd},
                           {"params": no_decay, "weight_decay": 0}],
                          lr=args.lr, momentum=0.9, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=args.lr, epochs=args.epochs, steps_per_epoch=len(train_loader),
        pct_start=0.15, div_factor=10, final_div_factor=1000)
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    # EMA of weights and BN buffers
    from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(args.ema), use_buffers=True) if args.ema > 0 else None
    eval_model = lambda: ema.module if ema is not None else model

    hist = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "lr": []}
    best, start = 0.0, 1
    ckpt_path = os.path.join(args.out, "last.pt")
    if args.resume and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        if ema is not None and ck.get("ema") is not None:
            ema.load_state_dict(ck["ema"])
        hist, best, start = ck["hist"], ck["best"], ck["epoch"] + 1
        print(f"Resumed from epoch {ck['epoch']} (best {best:.2f})")
    for ep in range(start, args.epochs + 1):
        model.train(); t0 = time.time()
        correct = total = 0; loss_sum = 0.0
        use_mix = ep <= args.epochs - args.mixup_off_epochs
        for x, y in train_loader:
            x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            mixed = use_mix and np.random.rand() < args.mixup_prob
            if mixed:
                x, ya, yb, lam = mixup(x, y)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                out = model(x)
                loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb) if mixed else criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update_parameters(model)
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); total += y.size(0)
        tl, ta = evaluate(eval_model(), test_loader, device) if has_val else (float("nan"), float("nan"))
        hist["train_loss"].append(loss_sum / total); hist["train_acc"].append(100 * correct / total)
        hist["test_loss"].append(tl); hist["test_acc"].append(ta); hist["lr"].append(sched.get_last_lr()[0])
        if not has_val or ta > best:
            best = ta if has_val else best
            torch.save(eval_model().state_dict(), os.path.join(args.out, "best_model.pt"))
        print(f"Ep {ep:3d}/{args.epochs} | train loss {loss_sum/total:.3f} acc {100*correct/total:.2f} "
              f"| val loss {tl:.3f} acc {ta:.2f} | best {best:.2f} | {time.time()-t0:.0f}s", flush=True)
        with open(os.path.join(args.out, "history.json"), "w") as f:
            json.dump(hist, f)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "ema": ema.state_dict() if ema is not None else None,
                    "hist": hist, "best": best, "epoch": ep}, ckpt_path)

    # ---- plots ----
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    e = range(1, len(hist["test_acc"]) + 1)
    ax[0].plot(e, hist["train_loss"], label="train"); ax[0].plot(e, hist["test_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(e, hist["train_acc"], label="train"); ax[1].plot(e, hist["test_acc"], label="val")
    ax[1].set_title("Accuracy (%)"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(os.path.join(args.out, "curves.png"), dpi=150)

    # ---- final eval + Kaggle submission ----
    model.load_state_dict(torch.load(os.path.join(args.out, "best_model.pt"), map_location=device, weights_only=True))
    if has_val:
        for mode in ("none", "flip", "full"):
            _, acc = evaluate(model, test_loader, device, tta=mode)
            print(f"Validation accuracy, TTA={mode:4s}: {acc:.2f}%")
        print(f"\nFINAL validation accuracy (held-out {args.val_size} imgs, full TTA): {acc:.2f}%")
    print(f"Number of trainable parameters: {n_params:,}")

    probs = []
    for i in range(0, len(x_kaggle), 512):
        x = torch.stack([TEST_TF(Image.fromarray(im)) for im in x_kaggle[i:i + 512]]).to(device)
        probs.append(predict_logits(model, x, device, tta="full").cpu())
    probs = torch.cat(probs).numpy()
    np.save(os.path.join(args.out, "test_probs.npy"), probs)   # for averaging several runs
    preds = probs.argmax(1)
    sub_path = os.path.join(args.out, "submission.csv")
    pd.DataFrame({"id": kaggle_ids, "label": preds}).to_csv(sub_path, index=False)
    print("Wrote", sub_path, "| prediction counts:", np.bincount(preds, minlength=10).tolist())


# ====== Kaggle settings: edit only this section ======
import glob, sys, shutil
matches = glob.glob('/kaggle/input/**/train_labels.csv', recursive=True)
if not matches:
    raise FileNotFoundError('Add the competition dataset via Kaggle > Add Input (train_labels.csv not found).')
DATA_DIR = os.path.dirname(matches[0])
FULL_TRAIN = False  # False: 45k train / 5k validation; True: all 50k for final submission
OUT = '/kaggle/working/improved_full' if FULL_TRAIN else '/kaggle/working/improved_validation'
sys.argv = ['notebook', '--data-dir', DATA_DIR, '--epochs', '240',
            '--val-size', '0' if FULL_TRAIN else '5000', '--workers', '2', '--out', OUT]
main()
shutil.copyfile(os.path.join(OUT, 'submission.csv'), '/kaggle/working/submission.csv')
print('Kaggle submission saved to /kaggle/working/submission.csv')


Trainable parameters: 4,327,754
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 32, 32]           1,728
       BatchNorm2d-2           [-1, 64, 32, 32]             128
              ReLU-3           [-1, 64, 32, 32]               0
            Conv2d-4           [-1, 64, 32, 32]          36,864
       BatchNorm2d-5           [-1, 64, 32, 32]             128
            Conv2d-6           [-1, 64, 32, 32]          36,864
       BatchNorm2d-7           [-1, 64, 32, 32]             128
        BasicBlock-8           [-1, 64, 32, 32]               0
            Conv2d-9           [-1, 64, 32, 32]          36,864
      BatchNorm2d-10           [-1, 64, 32, 32]             128
           Conv2d-11           [-1, 64, 32, 32]          36,864
      BatchNorm2d-12           [-1, 64, 32, 32]             128
       BasicBlock-13           [-1, 64, 32, 32]               0
       